<a href="https://colab.research.google.com/github/RonakkudalAI/Deep-Neural-Network/blob/main/01_finance_sentiment_analysis_using_bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Analysis using BERT

## Finance case study: understanding bank customer feedback

A bank receives customer feedback from mobile app reviews, email complaints, call center notes, and branch feedback forms. The customer support team cannot read every message immediately. Some messages are praise, some are neutral, and some are urgent complaints.

In this notebook, we will build an end-to-end sentiment analysis system using a Hugging Face transformer model. The system will read a customer message and classify it as:

- Negative
- Neutral
- Positive

The business goal is simple: identify unhappy customers faster so the bank can act before the customer leaves.


## Learning roadmap for this notebook

We will follow a complete project flow, not a collection of small code snippets.

1. Understand the banking business problem.
2. Create a realistic customer feedback dataset.
3. Explore the data like an analyst.
4. Learn what sentiment analysis means.
5. Learn what BERT means in simple language.
6. Understand tokenization and model inputs.
7. Fine-tune a transformer model using Hugging Face.
8. Evaluate the model.
9. Predict sentiment for new banking feedback.
10. Convert model output into business action.


## Code microscope: important Hugging Face words before we train

Before we run the model, let us slow down and understand the objects that will appear in the code.

| Term | Simple meaning | Why it matters |
|---|---|---|
| `model_checkpoint` | The name or path of a pretrained model. | It tells Hugging Face which model files to download or load. |
| Checkpoint | A saved model state containing learned weights, configuration, and sometimes tokenizer files. | We do not train from zero. We start from a model that already learned English patterns. |
| `AutoTokenizer` | A helper that automatically loads the correct tokenizer for the checkpoint. | Different models use different tokenization rules, so we should not guess manually. |
| Tokenizer | Converts text into token IDs and attention masks. | Neural networks read numbers, not raw words. |
| `AutoModelForSequenceClassification` | A helper that loads a transformer model with a classification layer on top. | We need one final label for each full message. |
| Sequence classification | Classifying an entire text sequence into one label. | Sentiment and sales routing are sequence classification tasks. |
| `logits` | Raw model scores before probabilities. | We use them to choose the class with the highest score. |
| `labels` | Correct answers from the dataset. | The Trainer compares predictions with labels during evaluation. |
| `np.argmax` | Selects the position of the largest value. | The largest logit becomes the predicted class. |
| `TrainingArguments` | A configuration object for training settings. | It controls epochs, batch size, learning rate, evaluation timing, and logging. |
| `Trainer` | Hugging Face's training manager. | It runs the training loop without us manually writing every PyTorch step. |


## 1. Why sentiment analysis matters in finance

Banks deal with trust. If a customer says, "My card was charged twice and nobody helped me", that message should not wait in a long queue.

Sentiment analysis helps the bank automatically detect the emotional tone of text.

For this case study:

- Negative feedback can be sent to a priority support queue.
- Neutral feedback can be stored for product analysis.
- Positive feedback can be used to understand what customers like.


## 2. Simple explanation of sentiment analysis

Sentiment analysis means teaching a computer to understand whether a piece of text sounds positive, negative, or neutral.

Real-world analogy:

Imagine a receptionist reading customer comments and placing each comment into one of three trays:

- "Happy customer"
- "Unhappy customer"
- "General information"

Our model will do a similar job automatically.


## 3. Tiny example

| Customer message | Expected sentiment |
|---|---|
| "The loan approval was quick and smooth." | Positive |
| "The app keeps crashing during payment." | Negative |
| "I visited the branch yesterday." | Neutral |

The model will learn patterns from examples like these.


In [ ]:
# This cell installs the main libraries used in this notebook.
# Run it once if you are using a fresh environment such as Google Colab.
# If the libraries are already installed, this cell may finish quickly.
%pip install -q transformers datasets accelerate evaluate scikit-learn pandas numpy matplotlib seaborn torch


In [ ]:
# pandas helps us create and analyze tabular datasets.
import pandas as pd

# numpy helps with numerical operations.
import numpy as np

# matplotlib is used for basic charts.
import matplotlib.pyplot as plt

# seaborn makes charts easier to read.
import seaborn as sns

# torch is the deep learning library used by Hugging Face models.
import torch

# train_test_split separates our data into training and testing parts.
from sklearn.model_selection import train_test_split

# LabelEncoder converts text labels such as "Positive" into numbers such as 2.
from sklearn.preprocessing import LabelEncoder

# classification_report gives precision, recall, F1-score, and accuracy.
from sklearn.metrics import classification_report

# confusion_matrix shows where the model is correct and where it is confused.
from sklearn.metrics import confusion_matrix

# Dataset is a Hugging Face object used by the Trainer API.
from datasets import Dataset

# AutoTokenizer loads the correct tokenizer for a selected model name.
from transformers import AutoTokenizer

# AutoModelForSequenceClassification loads a transformer model for classification.
from transformers import AutoModelForSequenceClassification

# TrainingArguments stores training settings such as epochs and batch size.
from transformers import TrainingArguments

# Trainer handles the training loop for transformer models.
from transformers import Trainer

# set_seed makes the experiment easier to reproduce.
from transformers import set_seed

# We set a seed so that the train-test split and model training are more stable.
set_seed(42)


## 4. Create a realistic banking feedback dataset

For a classroom lab, we will create a small dataset inside the notebook. In a real project, this data may come from:

- App store reviews
- Customer support tickets
- CRM notes
- Email feedback
- Call center transcripts

Each row will contain:

- `feedback_id`: unique feedback number
- `channel`: where the feedback came from
- `customer_feedback`: the actual text
- `sentiment`: the label we want the model to learn


In [ ]:
# We create a list of dictionaries.
# Each dictionary represents one customer feedback record.
bank_feedback_records = [
    {"feedback_id": 1, "channel": "Mobile App", "customer_feedback": "The mobile banking app crashes every time I try to transfer money.", "sentiment": "Negative"},
    {"feedback_id": 2, "channel": "Branch", "customer_feedback": "The branch staff helped me open my salary account very quickly.", "sentiment": "Positive"},
    {"feedback_id": 3, "channel": "Email", "customer_feedback": "I want to know the current interest rate for a home loan.", "sentiment": "Neutral"},
    {"feedback_id": 4, "channel": "Call Center", "customer_feedback": "My credit card was charged twice and nobody has resolved it yet.", "sentiment": "Negative"},
    {"feedback_id": 5, "channel": "Mobile App", "customer_feedback": "The new app design is clean and easy to use.", "sentiment": "Positive"},
    {"feedback_id": 6, "channel": "Branch", "customer_feedback": "I submitted my KYC documents at the branch today.", "sentiment": "Neutral"},
    {"feedback_id": 7, "channel": "Email", "customer_feedback": "The customer care executive was polite and solved my debit card issue.", "sentiment": "Positive"},
    {"feedback_id": 8, "channel": "Mobile App", "customer_feedback": "The UPI payment failed but the amount was deducted from my account.", "sentiment": "Negative"},
    {"feedback_id": 9, "channel": "Call Center", "customer_feedback": "Please send me the statement for the last six months.", "sentiment": "Neutral"},
    {"feedback_id": 10, "channel": "Branch", "customer_feedback": "The loan officer explained every step clearly and patiently.", "sentiment": "Positive"},
    {"feedback_id": 11, "channel": "Mobile App", "customer_feedback": "I cannot log in even after resetting my password three times.", "sentiment": "Negative"},
    {"feedback_id": 12, "channel": "Email", "customer_feedback": "I would like to update my registered mobile number.", "sentiment": "Neutral"},
    {"feedback_id": 13, "channel": "Call Center", "customer_feedback": "Your fraud alert saved me from an unauthorized transaction.", "sentiment": "Positive"},
    {"feedback_id": 14, "channel": "Mobile App", "customer_feedback": "The app shows incorrect account balance after every transaction.", "sentiment": "Negative"},
    {"feedback_id": 15, "channel": "Branch", "customer_feedback": "I collected my checkbook from the branch.", "sentiment": "Neutral"},
    {"feedback_id": 16, "channel": "Email", "customer_feedback": "The fixed deposit booking process was simple and fast.", "sentiment": "Positive"},
    {"feedback_id": 17, "channel": "Call Center", "customer_feedback": "I waited on hold for forty minutes and still did not get help.", "sentiment": "Negative"},
    {"feedback_id": 18, "channel": "Mobile App", "customer_feedback": "I need information about international debit card charges.", "sentiment": "Neutral"},
    {"feedback_id": 19, "channel": "Branch", "customer_feedback": "The relationship manager gave excellent advice for my savings plan.", "sentiment": "Positive"},
    {"feedback_id": 20, "channel": "Email", "customer_feedback": "My loan application status has not changed for two weeks.", "sentiment": "Negative"},
    {"feedback_id": 21, "channel": "Mobile App", "customer_feedback": "The account summary page loads instantly now.", "sentiment": "Positive"},
    {"feedback_id": 22, "channel": "Call Center", "customer_feedback": "I want to block my old debit card and request a new one.", "sentiment": "Neutral"},
    {"feedback_id": 23, "channel": "Branch", "customer_feedback": "The cash deposit machine was not working again.", "sentiment": "Negative"},
    {"feedback_id": 24, "channel": "Email", "customer_feedback": "Thank you for reversing the incorrect penalty charge.", "sentiment": "Positive"},
    {"feedback_id": 25, "channel": "Mobile App", "customer_feedback": "I am checking whether my nominee details are updated.", "sentiment": "Neutral"},
    {"feedback_id": 26, "channel": "Call Center", "customer_feedback": "The agent disconnected the call before answering my question.", "sentiment": "Negative"},
    {"feedback_id": 27, "channel": "Branch", "customer_feedback": "The branch queue system was organized and fast.", "sentiment": "Positive"},
    {"feedback_id": 28, "channel": "Email", "customer_feedback": "Please confirm the minimum balance requirement for my account.", "sentiment": "Neutral"},
    {"feedback_id": 29, "channel": "Mobile App", "customer_feedback": "The bill payment feature is convenient and reliable.", "sentiment": "Positive"},
    {"feedback_id": 30, "channel": "Call Center", "customer_feedback": "I am frustrated because my complaint has been reopened three times.", "sentiment": "Negative"},
]

# We convert the list of dictionaries into a pandas DataFrame.
feedback_df = pd.DataFrame(bank_feedback_records)


# We display the first few rows to inspect the dataset.
feedback_df.head()




## 5. Dataset understanding

Before training any model, always understand the data.

Good data understanding helps us answer questions such as:

- Do we have enough examples for each sentiment?
- Are the feedback messages short or long?
- Are some channels producing more complaints than others?


In [ ]:
# shape tells us the number of rows and columns.
feedback_df.shape


In [ ]:
# value_counts counts how many examples are available for each sentiment label.
feedback_df["sentiment"].value_counts()


In [ ]:
# We create a new column that stores the number of words in each feedback message.
feedback_df["word_count"] = feedback_df["customer_feedback"].str.split().str.len()

# We calculate basic statistics for the word counts.
feedback_df["word_count"].describe()


In [ ]:
# We set the chart size so the plot is easy to read.
plt.figure(figsize=(7, 4))

# We plot the number of examples in each sentiment class.
sns.countplot(data=feedback_df, x="sentiment", order=["Negative", "Neutral", "Positive"])

# We add a title to explain what the chart shows.
plt.title("Number of customer feedback examples by sentiment")

# We label the x-axis.
plt.xlabel("Sentiment")

# We label the y-axis.
plt.ylabel("Number of feedback messages")

# We display the chart.
plt.show()


## 6. What is BERT?

BERT stands for Bidirectional Encoder Representations from Transformers.

That sounds heavy, so let us break it down.

### What does "bidirectional" mean?

Bidirectional means BERT looks at words on both sides of a word.

Example:

Sentence 1: "I deposited cash in the bank."

Sentence 2: "We sat near the river bank."

The word "bank" has different meanings. BERT looks at surrounding words such as "deposited", "cash", "river", and "sat" to understand the correct meaning.

### What does "encoder" mean?

An encoder converts input text into useful numerical representations. These numbers help the model understand meaning, context, and relationships between words.

### Why use BERT for sentiment?

BERT is useful because it understands context. In finance, context matters a lot.

Example:

"The charge was reversed after three weeks."

This may sound positive because the charge was reversed, but it may also show frustration because it took three weeks. A context-aware model is better than a simple keyword-based system.


## 7. Advantages and disadvantages of BERT

### Advantages

- Understands context better than many older NLP methods.
- Can be fine-tuned for many tasks such as sentiment analysis, text classification, and question answering.
- Works well even when the dataset is not extremely large, because it already learned language patterns during pretraining.

### Disadvantages

- Larger than traditional machine learning models.
- Slower to train than models such as logistic regression.
- Needs more memory and compute.
- Can still make mistakes when text is sarcastic, vague, or very domain-specific.

For this classroom notebook, we will use DistilBERT. DistilBERT is a smaller and faster version inspired by BERT. It is easier for students to run on normal machines.


## 8. Prepare labels for modeling

Machine learning models do not directly understand labels like `Negative`, `Neutral`, and `Positive`.

We convert them into numbers:

- Negative -> 0
- Neutral -> 1
- Positive -> 2

This is called label encoding.


In [ ]:
# We create a LabelEncoder object.
label_encoder = LabelEncoder()

# We fit the encoder on the sentiment column and create numeric labels.
feedback_df["label"] = label_encoder.fit_transform(feedback_df["sentiment"])

# We display the label mapping so students can understand the conversion.
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

# We print the mapping.
label_mapping

In [ ]:
# We keep only the columns needed for model training.
model_df = feedback_df[["customer_feedback", "label"]].copy()


# We rename the text column to "text" because this is a common Hugging Face convention.
model_df = model_df.rename(columns={"customer_feedback": "text"})


# We display the prepared modeling dataset.
model_df.head()


## 9. Train-test split

We train the model on one part of the data and test it on another part.

This helps us check whether the model learned useful patterns instead of only memorizing the training examples.


In [ ]:
# We split the dataset into training and testing data.
# stratify keeps the class distribution similar in both sets.
train_df, test_df = train_test_split(
    model_df,
    test_size=0.25,
    random_state=42,
    stratify=model_df["label"],
)


# We reset the row index so Hugging Face can read the data cleanly.
train_df = train_df.reset_index(drop=True)

# We reset the row index for the test data too.
test_df = test_df.reset_index(drop=True)

# We print the size of each split.
print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))


## 10. Tokenization

Transformer models do not read raw text directly.

They first convert text into tokens and token IDs.

Tiny example:

Text: "The app is slow"

Possible tokens: `["the", "app", "is", "slow"]`

Possible token IDs: `[1996, 10439, 2003, 4030]`

The exact IDs depend on the tokenizer vocabulary.

The tokenizer also creates an attention mask. The attention mask tells the model which tokens are real text and which tokens are padding.


In [ ]:
# A checkpoint is the name or location of saved pretrained model files.
# Here, "distilbert-base-uncased" is a public model checkpoint on the Hugging Face Hub.
# "distilbert" means it is a smaller, faster version inspired by BERT.
# "base" means it is the standard-size DistilBERT model.
# "uncased" means the model treats "Bank", "bank", and "BANK" mostly the same by lowercasing text.
model_checkpoint = "distilbert-base-uncased"

# AutoTokenizer is a smart loader.
# Instead of manually choosing a tokenizer class, we give Hugging Face the checkpoint name.
# Hugging Face reads the checkpoint configuration and loads the tokenizer that matches this model.
# This matters because BERT, DistilBERT, RoBERTa, and T5 may tokenize text differently.
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# We choose one real feedback message from our finance dataset.
# loc[0, "customer_feedback"] means: row index 0, column named "customer_feedback".
sample_text = feedback_df.loc[0, "customer_feedback"]

# The tokenizer converts text into a dictionary of model inputs.
# The most important keys are usually:
# - input_ids: numeric IDs for tokens.
# - attention_mask: 1 for real tokens and 0 for padding tokens.
sample_tokens = tokenizer(sample_text)

# We display the original human-readable text.
print("Original text:")
print(sample_text)

# input_ids are the numbers the model will actually receive.
# Each number points to a token in the tokenizer vocabulary.
print("\nToken IDs:")
print(sample_tokens["input_ids"])

# convert_ids_to_tokens changes the numeric IDs back into token strings.
# This helps students see how the sentence was broken into smaller pieces.
print("\nTokens:")
print(tokenizer.convert_ids_to_tokens(sample_tokens["input_ids"]))


In [ ]:
# Hugging Face Trainer works best with Hugging Face Dataset objects.
# Dataset.from_pandas converts our pandas DataFrame into that format.
train_dataset = Dataset.from_pandas(train_df)

# We convert the test DataFrame in the same way.
test_dataset = Dataset.from_pandas(test_df)

# This function receives a batch, not just one row.
# A batch is a small group of examples processed together for speed.
def tokenize_batch(batch):
    # batch["text"] contains several feedback messages from the dataset.
    # tokenizer(...) converts those messages into input_ids and attention_mask.
    return tokenizer(
        batch["text"],

        # padding="max_length" forces every example to have the same length.
        # Models train faster when examples in a batch have equal shape.
        padding="max_length",

        # truncation=True cuts text that is longer than max_length.
        # Without truncation, very long text can exceed the model limit and cause an error.
        truncation=True,

        # max_length=96 means every feedback message becomes 96 token positions.
        # Short messages are padded. Long messages are cut to 96 tokens.
        max_length=96,
    )

# map applies tokenize_batch to the full training dataset.
# batched=True means Hugging Face sends multiple rows at once to the function.
tokenized_train = train_dataset.map(tokenize_batch, batched=True)

# We apply the exact same tokenization process to the test dataset.
tokenized_test = test_dataset.map(tokenize_batch, batched=True)

# with_format("torch") tells the dataset to return PyTorch tensors.
# PyTorch tensors are the numeric format expected by the transformer model.
tokenized_train = tokenized_train.with_format("torch")

# We also format the test dataset as PyTorch tensors.
tokenized_test = tokenized_test.with_format("torch")


## 11. Load the model

We use `AutoModelForSequenceClassification`.

Sequence classification means the model reads one full text sequence and predicts one label for the whole sequence.

In this notebook, each customer feedback message gets one sentiment label.


In [ ]:
# num_labels tells the model how many output classes it must predict.
# In this finance notebook, the labels are Negative, Neutral, and Positive.
num_labels = feedback_df["label"].nunique()

# AutoModelForSequenceClassification is a Hugging Face auto-class.
# "Auto" means Hugging Face will inspect the checkpoint and choose the correct model architecture.
# "SequenceClassification" means the model will classify one complete text sequence into one label.
# from_pretrained(...) loads learned language weights from the checkpoint instead of starting randomly.
model = AutoModelForSequenceClassification.from_pretrained(
    # model_checkpoint points to the pretrained DistilBERT files.
    model_checkpoint,

    # num_labels creates a classification output layer with exactly this many classes.
    # The base DistilBERT model understands text, but this task-specific head predicts sentiment.
    num_labels=num_labels,
)


In [ ]:
# Trainer calls this function during evaluation.
# eval_prediction is created by Trainer after the model predicts on the evaluation dataset.
def compute_metrics(eval_prediction):
    # eval_prediction contains two main things:
    # 1. predictions: raw model outputs, usually called logits.
    # 2. label_ids: the correct labels from the dataset.
    logits, labels = eval_prediction

    # logits are raw scores produced by the model before softmax.
    # Example for one message with three classes: [1.2, -0.4, 2.8]
    # These are not probabilities yet, but the largest score usually represents the predicted class.

    # np.argmax(logits, axis=-1) finds the index of the largest logit for each row.
    # axis=-1 means "look across the class scores for each example".
    # If logits are [1.2, -0.4, 2.8], argmax returns 2.
    predictions = np.argmax(logits, axis=-1)

    # labels are the correct answer IDs from our test/evaluation data.
    # predictions == labels creates True/False values for correct and incorrect predictions.
    # mean() converts True to 1 and False to 0, then calculates the average correctness.
    accuracy = (predictions == labels).mean()

    # Trainer expects a dictionary of metric names and metric values.
    # We return accuracy so it appears in the training logs.
    return {"accuracy": accuracy}


## 12. Fine-tune the model

Fine-tuning means we start with a model that already knows general English and then train it a little more on our banking feedback data.

For classroom speed, we use a small number of epochs. In a real bank project, we would use more data, a validation set, stronger evaluation, and careful monitoring.


In [ ]:
# output_directory is the folder where Trainer can save training artifacts.
# In this notebook, we are not saving checkpoints permanently, but Trainer still needs an output path.
output_directory = "./finance_sentiment_model"

# TrainingArguments is a configuration object.
# It does not train the model by itself.
# It simply stores all the settings that Trainer will use during training.
training_args = TrainingArguments(
    # output_dir tells Trainer where to write logs, temporary files, and model outputs.
    output_dir=output_directory,

    # num_train_epochs means how many full passes the model makes over the training dataset.
    # 2 epochs means the model sees every training example two times.
    num_train_epochs=2,

    # per_device_train_batch_size is the number of training examples processed at once on each device.
    # A device can be a CPU, GPU, or accelerator.
    # Smaller batch sizes use less memory and are safer for student laptops.
    per_device_train_batch_size=4,

    # per_device_eval_batch_size is the number of evaluation examples processed at once per device.
    # It can be larger than training batch size, but we keep it small for reliability.
    per_device_eval_batch_size=4,

    # learning_rate controls how large each model update step is.
    # 2e-5 means 0.00002, a common starting value for fine-tuning BERT-style models.
    learning_rate=2e-5,

    # eval_strategy="epoch" tells Trainer to evaluate after every epoch.
    # This lets us see whether the model is improving after each full pass over the data.
    eval_strategy="epoch",

    # save_strategy="no" avoids saving model checkpoints after every epoch.
    # This keeps the classroom notebook lighter and avoids creating many files.
    save_strategy="no",

    # logging_steps controls how often training progress is printed.
    # With a tiny dataset, a small number helps students see progress.
    logging_steps=5,

    # report_to="none" disables external experiment trackers such as Weights & Biases.
    # This avoids login prompts during training.
    report_to="none",
)

# Trainer is Hugging Face's high-level training manager.
# It connects the model, data, training settings, tokenizer, and metric function.
trainer = Trainer(
    # model is the DistilBERT classifier we loaded above.
    model=model,

    # args contains all training settings such as epochs and batch size.
    args=training_args,

    # train_dataset is the tokenized data used to update model weights.
    train_dataset=tokenized_train,

    # eval_dataset is the tokenized data used to measure performance after each epoch.
    eval_dataset=tokenized_test,

    # tokenizer is passed so Trainer knows how inputs were prepared.
    # It may also be used when saving model artifacts.
    tokenizer=tokenizer,

    # compute_metrics tells Trainer how to calculate evaluation metrics from predictions.
    compute_metrics=compute_metrics,
)

# trainer.train() starts the fine-tuning loop.
# During training, the model compares predictions with labels and adjusts its weights.
trainer.train()


## 13. Evaluate the model

Evaluation tells us how well the model performs on unseen test data.

For sentiment analysis, common metrics are:

- Accuracy: overall percentage of correct predictions.
- Precision: when the model predicts a class, how often is it correct?
- Recall: out of all real examples of a class, how many did the model find?
- F1-score: a balance between precision and recall.


In [ ]:
# trainer.predict runs the trained model on the tokenized test dataset.
# It returns a PredictionOutput object containing predictions, labels, and metrics.
test_predictions = trainer.predict(tokenized_test)

# predictions are raw model outputs called logits.
# Shape is usually: number_of_examples x number_of_classes.
logits = test_predictions.predictions

# For each test example, we select the class with the largest logit.
# The result is a list of numeric label IDs such as [0, 2, 1].
predicted_label_ids = np.argmax(logits, axis=1)

# These are the correct numeric labels from our test DataFrame.
true_label_ids = test_df["label"].values

# The model predicts numbers, but humans need readable labels.
# inverse_transform converts label IDs back into names such as Negative, Neutral, Positive.
predicted_labels = label_encoder.inverse_transform(predicted_label_ids)

# We also convert the true label IDs into readable label names.
true_labels = label_encoder.inverse_transform(true_label_ids)

# classification_report prints precision, recall, F1-score, and accuracy.
# This gives a more detailed view than accuracy alone.
print(classification_report(true_labels, predicted_labels))


In [ ]:
# We create a confusion matrix to see mistakes class by class.
cm = confusion_matrix(true_labels, predicted_labels, labels=label_encoder.classes_)

# We set the chart size.
plt.figure(figsize=(6, 4))

# We draw the confusion matrix as a heatmap.
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
    cmap="Blues",
)

# We add chart labels.
plt.xlabel("Predicted sentiment")
plt.ylabel("Actual sentiment")
plt.title("Confusion matrix for banking sentiment model")

# We show the chart.
plt.show()


## 14. Predict sentiment for new banking feedback

Now we will create a reusable function.

The function will:

1. Take a new customer message.
2. Tokenize it.
3. Send it to the model.
4. Convert model scores into probabilities.
5. Return the predicted sentiment.


In [ ]:
# This function predicts sentiment for one new customer message.
def predict_sentiment(customer_message):
    # customer_message is a normal Python string written by a user or received from a business system.

    # We tokenize the new message using the same tokenizer used during training.
    # return_tensors="pt" means return PyTorch tensors.
    # truncation=True prevents very long messages from breaking the model limit.
    # padding=True adds padding if needed so the tensor shape is valid.
    encoded_input = tokenizer(
        customer_message,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=96,
    )

    # The model may be on CPU or GPU.
    # This line moves every input tensor to the same device as the model.
    encoded_input = {key: value.to(model.device) for key, value in encoded_input.items()}

    # torch.no_grad() tells PyTorch not to calculate gradients.
    # Gradients are needed for training, but not for prediction.
    # This makes prediction faster and uses less memory.
    with torch.no_grad():
        # model(**encoded_input) sends input_ids and attention_mask into the model.
        # The ** syntax unpacks the dictionary into named arguments.
        output = model(**encoded_input)

    # output.logits contains raw scores for every sentiment class.
    # softmax converts raw logits into probabilities that add up to 1.
    probabilities = torch.softmax(output.logits, dim=1).cpu().numpy()[0]

    # np.argmax selects the class ID with the highest probability.
    predicted_id = int(np.argmax(probabilities))

    # inverse_transform converts the numeric ID into a readable sentiment label.
    predicted_sentiment = label_encoder.inverse_transform([predicted_id])[0]

    # This dictionary shows the probability for each sentiment label.
    # It helps us understand whether the model is confident or uncertain.
    probability_table = {
        label_encoder.inverse_transform([class_id])[0]: float(probability)
        for class_id, probability in enumerate(probabilities)
    }

    # We return the final sentiment and the probability details.
    return predicted_sentiment, probability_table


In [ ]:
# We create realistic new messages that were not part of the training data.
new_feedback_examples = [
    "The mobile app deducted money but the merchant did not receive the payment.",
    "The branch manager helped me complete my education loan documentation.",
    "Please tell me the documents required for a personal loan.",
]

# We loop through each message and predict the sentiment.
for message in new_feedback_examples:
    # We call our prediction function.
    sentiment, probabilities = predict_sentiment(message)

    # We print the original message.
    print("Customer message:", message)

    # We print the predicted sentiment.
    print("Predicted sentiment:", sentiment)

    # We print class probabilities.
    print("Probabilities:", probabilities)

    # We print a separator line.
    print("-" * 80)


## 15. Business interpretation

The model prediction is useful only when it leads to action.

Example business rules:

- Negative -> create high-priority support ticket.
- Neutral -> route to information or operations queue.
- Positive -> store as customer experience insight.

This is how a machine learning output becomes a business workflow.


In [ ]:
# This function converts sentiment into a suggested business action.
def recommend_action(sentiment):
    # Negative feedback usually needs urgent support.
    if sentiment == "Negative":
        return "Create high-priority support ticket"

    # Neutral feedback usually needs normal processing or information response.
    if sentiment == "Neutral":
        return "Route to standard service queue"

    # Positive feedback can be used for experience analysis.
    return "Store as positive customer experience insight"

# We create an empty list to store business outputs.
business_outputs = []

# We process each new customer message.
for message in new_feedback_examples:
    # We predict sentiment.
    sentiment, probabilities = predict_sentiment(message)

    # We create a business action.
    action = recommend_action(sentiment)

    # We append the result as a dictionary.
    business_outputs.append(
        {
            "customer_feedback": message,
            "predicted_sentiment": sentiment,
            "recommended_action": action,
        }
    )

# We convert the business outputs into a DataFrame.
business_output_df = pd.DataFrame(business_outputs)

# We display the business output table.
business_output_df


## 16. Limitations and risks

This model is useful, but it is not perfect.

Possible limitations:

- Small training datasets may lead to weak performance.
- Sarcasm is difficult. Example: "Great, the app failed again."
- Financial terms may need domain-specific training data.
- Customer data may contain sensitive information.
- Wrong predictions can create poor customer experience.

In a real banking project, human review should remain part of the workflow, especially for complaints, fraud, loans, and regulated communication.


## 17. Alternatives

Other approaches we could use:

- Rule-based keywords: simple but weak for complex language.
- TF-IDF with logistic regression: fast and strong baseline.
- Word2Vec or GloVe with machine learning: useful but less context-aware.
- LSTM or GRU: good sequence models, but usually weaker than transformers on many NLP tasks.
- Larger transformer models such as RoBERTa or DeBERTa: potentially better but heavier.
- LLM APIs: powerful, but cost, privacy, and governance must be considered.


## 18. Student practice

Try these tasks:

1. Add 15 more banking feedback examples.
2. Add a new label called `Urgent Negative`.
3. Test the model on polite complaints and angry complaints.
4. Compare DistilBERT with `bert-base-uncased`.
5. Write three business rules for routing customer feedback.

## Final takeaway

You built an end-to-end finance sentiment analysis case study using Hugging Face. You moved from business problem to dataset, model training, evaluation, prediction, and business action.
